In [4]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from typing import Dict, List, Tuple 
import logging

logger = logging.getLogger(__name__)

spark = SparkSession.builder.getOrCreate()



In [5]:
def check_nulls(df:DataFrame, required_cols: List[str]) -> Tuple[bool, Dict[str, int]]:
        """ 
        Check for null values in the specified fields

        Args:
            df (DataFrame): Dataframe with data to be checked
            required_cols (List[str]): List of columns to check for null values

        Returns:
            Tuple[bool, Dict[str, int]]: (passed/failed, dict of null counts by column)
        """
        logger.info(f"checking for null columns in: {required_cols}")

        null_counts = {}
        for col in required_cols:
            if col in df.columns:
                null_count = df.filter(F.col(col).isNull()).count()
                null_counts[col] = null_count
            else:
                logger.warning(f"Column {col} not found in the DataFrame")
                null_counts[col] = "Column not found"

        has_nulls = any(isinstance(count, int) and count > 0 for count in null_counts.values())

        if has_nulls:
             logger.warning(f"Null check failed. Null counts: {null_counts}")
             return False, null_counts
        else:
             logger.info("Null check passed")
             return True, null_counts 
        


In [31]:
df = spark.read.csv("../data/raw/transactions/", header=True)

df_accounts = spark.read.csv("../data/raw/accounts/", header=True)

In [9]:
cols = df.columns

In [11]:
check_nulls(df, cols)

(True,
 {'transaction_id': 0,
  'account_id': 0,
  'transaction_date': 0,
  'transaction_type': 0,
  'amount': 0,
  'currency': 0,
  'description': 0,
  'merchant_name': 0,
  'merchant_category': 0,
  'transaction_status': 0,
  'channel': 0,
  'location': 0,
  'is_international': 0})

In [12]:
def check_duplicates(df:DataFrame, key_columns: List[str]) -> Tuple[bool, int]:
    """ 
    Check for duplicate records based on keys

    Args:
        df (DataFrame): Dataframe with data to be checked

        key_columns(List[str]): columns that should compose a unique key

    Returns:
        Tuple[bool, int]: (passsed/failed, count of duplicate records)
    """
    logger.info(f"Checking for duplicates on key columns")

    total_rows = df.count()

    distinct_rows = df.select(key_columns).distinct().count()

    duplicate_count = total_rows - distinct_rows

    if duplicate_count > 0:
        logger.warning(f"Duplicate check failed. Found {duplicate_count} duplicates")
        return False, duplicate_count
    else:
        logger.info("Duplicate check has passed")
        return True, 0
    


In [14]:
cols

['transaction_id',
 'account_id',
 'transaction_date',
 'transaction_type',
 'amount',
 'currency',
 'description',
 'merchant_name',
 'merchant_category',
 'transaction_status',
 'channel',
 'location',
 'is_international']

In [24]:
check_duplicates(df, ['amount'])

Duplicate check failed. Found 7 duplicates


(False, 7)

In [16]:
df1 = df.union(df)

In [17]:
df1.count()

20000

In [18]:
df.count()

10000

In [23]:
check_duplicates(df, ['transaction_date'])

Duplicate check failed. Found 16 duplicates


(False, 16)

In [25]:
def check_data_ranges(df: DataFrame, range_checks: Dict[str, Tuple]) -> Tuple[bool, Dict[str, int]]:
    """ 
    Check if values in column fall in expected range

    Args:
        df (DataFrame): Dataframe to check
        range_checks (Dict[str, Tuple]): Dictionary mapping columns to range of values

    Returns:
        Tuple[bool, Dict[str, int]]: (passed/failed, dict of out of range counts by column)
    """
    logger.info(f"Checking data ranges for columns: {list(range_checks.keys())}")

    out_of_range_counts = {}

    for col, (min_value, max_value) in range_checks.items():
        if col in df.columns:
            out_of_range_count = df.filter(
                (F.col(col) < min_value) | (F.col(col) > max_value)
            ).count()

            out_of_range_counts[col] = out_of_range_count
        else:
            logger.warning(f"Column {col} not found")
            out_of_range_counts[col] = "Column not found"

    has_out_of_range = any(isinstance(count, int) and count > 0 for count in out_of_range_counts.values())

    if has_out_of_range:
        logger.warning(f"Range check failed. Out of range counts: {out_of_range_counts}")
        return False, out_of_range_counts
    else:
        logger.info("Range check passed")
        return True, out_of_range_counts



In [28]:
check_data_ranges(df, {"amount": (0, 2000)})

Range check failed. Out of range counts: {'amount': 5989}


(False, {'amount': 5989})

In [30]:
def check_referential_integrity(df:DataFrame, ref_df:DataFrame, fk_column: str, pk_column: str) -> Tuple[bool, int]:
    """ 
    Check referential integrity between two dataframes

    Args:
        df (DataFrame): Dataframe with foreign key
        ref_df (DataFrame): DataFrame with primary key
        fk_column (str): Foreign key column in df dataframe
        pk_column (str): Promary key column in ref_df dataframe

    Returns:
        Tuple[bool, int]: (passed/failed, count of orphaned records)
    """

    logger.info(f"Checking referential integrity: {fk_column} in {pk_column}")

    fk_values = df.select(fk_column).distinct()

    pk_values = ref_df.select(pk_column).distinct()

    orphaned_records = fk_values.exceptAll(pk_values)

    orphaned_count = orphaned_records.count()

    if orphaned_count > 0:
        logger.warning(f"Referential integrity check failed. Found {orphaned_count} orphaned records")
        return False, orphaned_count
    else:
        logger.info("Referential integrity check passed")
        return True, 0
    





In [33]:
check_referential_integrity(df, df_accounts, 'account_id', 'account_id')

(True, 0)